# 🚌 Public Transport Demand Predictor
## Machine Learning Notebook — Google Colab Edition

**SDG 11 – Sustainable Cities and Communities**

> ⚠️ **DISCLAIMER**: This notebook uses a **synthetic** dataset.
> It does **not** represent real passenger observations.

---
### Contents
1. Install dependencies
2. Generate synthetic dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Model Training & Evaluation
6. Results & Visualisation
7. Making Predictions


## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install scikit-learn pandas numpy matplotlib plotly joblib --quiet
print('✅ All packages installed')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import joblib, json, warnings
warnings.filterwarnings('ignore')

from sklearn.dummy        import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree         import DecisionTreeRegressor
from sklearn.ensemble     import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics      import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
rng  = np.random.default_rng(SEED)
print('✅ Imports complete')

## 2. Generate Synthetic Dataset

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
ROUTES   = ['R01', 'R02', 'R03', 'R04', 'R05']
WEATHERS = ['Clear', 'Cloudy', 'Rainy', 'Foggy']
ROUTE_BASE = {'R01': 120, 'R02': 95, 'R03': 150, 'R04': 80, 'R05': 110}
HOLIDAYS = {
    '2023-01-26','2023-08-15','2023-10-02','2023-12-25',
    '2024-01-26','2024-08-15','2024-10-02','2024-12-25',
}

def hour_multiplier(h):
    peaks = {6:0.60,7:0.85,8:1.40,9:1.30,10:0.90,11:0.80,12:0.95,
             13:0.90,14:0.75,15:0.80,16:1.00,17:1.35,18:1.50,19:1.25,
             20:1.00,21:0.75,22:0.50,23:0.30}
    return peaks.get(h, 0.20)

def weather_mult(w):
    return {'Clear':1.0,'Cloudy':0.95,'Rainy':0.80,'Foggy':0.88}[w]

records = []
for date in pd.date_range('2023-01-01','2024-12-31',freq='D'):
    ds  = date.strftime('%Y-%m-%d')
    dow = date.day_name()
    iwe = int(date.weekday() >= 5)
    iho = int(ds in HOLIDAYS)
    doy = date.day_of_year
    temp = round(float(np.clip(25 + 10*np.sin(2*np.pi*(doy-80)/365) + rng.normal(0,2), 10, 45)), 1)
    wth = rng.choice(WEATHERS, p=[0.50,0.25,0.15,0.10])
    for hour in range(6, 24):
        for route in ROUTES:
            cnt = max(0, int(round(
                ROUTE_BASE[route] * hour_multiplier(hour) * weather_mult(wth)
                * (0.75 if iwe else 1.0) * (0.65 if iho else 1.0)
                * rng.normal(1.0, 0.12)
            )))
            records.append([ds, route, hour, dow, iwe, temp, wth, iho, cnt])

df = pd.DataFrame(records,
    columns=['Date','Route_ID','Hour','Day_of_Week','Is_Weekend',
             'Temperature','Weather','Is_Holiday','Passenger_Count'])

# Inject missing values, duplicates, outliers
for col in ['Temperature','Weather']:
    df.loc[rng.choice(len(df),30,replace=False), col] = np.nan
dups = df.iloc[rng.choice(len(df),20,replace=False)].copy()
df = pd.concat([df, dups], ignore_index=True)
df.loc[rng.choice(len(df),10,replace=False), 'Passenger_Count'] = rng.integers(500,800,10)

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('--- Basic Info ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Descriptive Statistics ---')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('EDA: Passenger Count Distributions', fontweight='bold')

# Histogram
axes[0].hist(df['Passenger_Count'].dropna(), bins=40, color='#1976D2', alpha=0.8, edgecolor='white')
axes[0].set_title('Passenger Count Distribution')
axes[0].set_xlabel('Passenger Count'); axes[0].set_ylabel('Frequency')

# Average by route
route_avg = df.groupby('Route_ID')['Passenger_Count'].mean()
axes[1].bar(route_avg.index, route_avg.values, color='#43A047', edgecolor='white')
axes[1].set_title('Average Passengers by Route')

# Average by hour
hour_avg = df.groupby('Hour')['Passenger_Count'].mean()
axes[2].plot(hour_avg.index, hour_avg.values, 'o-', color='#E91E63', lw=2)
axes[2].set_title('Average Passengers by Hour')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# 4a. Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=['Date','Route_ID','Hour'], keep='first').reset_index(drop=True)
print(f'Duplicates removed: {before - len(df)}')

# 4b. Handle missing values
df['Date'] = pd.to_datetime(df['Date'])
df['Temperature'] = df.groupby(['Route_ID','Hour'])['Temperature'] \
                      .transform(lambda s: s.fillna(s.median()))
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
df['Weather']     = df['Weather'].fillna('Clear')
print(f'Missing after fill: {df.isnull().sum().sum()}')

# 4c. Outlier capping (IQR × 3)
Q1, Q3 = df['Passenger_Count'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lo, hi = Q1 - 3*IQR, Q3 + 3*IQR
n_cap = ((df['Passenger_Count'] < lo) | (df['Passenger_Count'] > hi)).sum()
df['Passenger_Count'] = df['Passenger_Count'].clip(lo, hi)
print(f'Outliers capped: {n_cap}')

# 4d. Feature engineering
df['Month']      = df['Date'].dt.month
df['Quarter']    = df['Date'].dt.quarter
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['Hour_sin']   = np.sin(2*np.pi*df['Hour']/24)
df['Hour_cos']   = np.cos(2*np.pi*df['Hour']/24)
df['Month_sin']  = np.sin(2*np.pi*df['Month']/12)
df['Month_cos']  = np.cos(2*np.pi*df['Month']/12)

# 4e. Label encoding
encoders = {}
for col in ['Route_ID','Day_of_Week','Weather']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'Encoded {col}: {list(le.classes_)}')

print(f'\nFinal shape: {df.shape}')

In [ ]:
# 4f. Chronological train/test split (no data leakage)
FEATURES = ['Route_ID','Hour','Hour_sin','Hour_cos','Day_of_Week',
            'Is_Weekend','Temperature','Weather','Is_Holiday',
            'Month','Month_sin','Month_cos','Quarter','WeekOfYear']
TARGET   = 'Passenger_Count'

df_sorted = df.sort_values('Date').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.80)
train, test = df_sorted.iloc[:split_idx], df_sorted.iloc[split_idx:]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')
print(f'Train dates: {train["Date"].min().date()} → {train["Date"].max().date()}')
print(f'Test  dates: {test["Date"].min().date()}  → {test["Date"].max().date()}')

## 5. Model Training & Evaluation

In [ ]:
def evaluate(name, model, X_test, y_test):
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    return {'Model': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R2': round(r2,4)}

models = {
    'Baseline (Mean)':  DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Decision Tree':    DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=SEED),
    'Random Forest':    RandomForestRegressor(n_estimators=150, max_depth=15,
                                               min_samples_leaf=5, n_jobs=-1, random_state=SEED),
}

results = []
fitted  = {}
for name, m in models.items():
    print(f'Training {name}...')
    m.fit(X_train, y_train)
    r = evaluate(name, m, X_test, y_test)
    results.append(r)
    fitted[name] = m
    print(f'  MAE={r["MAE"]}  RMSE={r["RMSE"]}  R²={r["R2"]}')

metrics_df = pd.DataFrame(results)
metrics_df

In [ ]:
# Select best model (lowest RMSE, excluding baseline)
non_base = metrics_df[metrics_df['Model'] != 'Baseline (Mean)']
best_row = non_base.loc[non_base['RMSE'].idxmin()]
best_model = fitted[best_row['Model']]
print(f'✅ Best model: {best_row["Model"]}  (RMSE={best_row["RMSE"]})')

# Save model
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(encoders,   'label_encoders.pkl')
print('Models saved to disk.')

## 6. Results & Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontweight='bold')
names = metrics_df['Model']
x = range(len(names))
for ax, metric, col in zip(axes, ['MAE','RMSE','R2'], ['#2196F3','#4CAF50','#FF9800']):
    bars = ax.bar(x, metrics_df[metric], color=col, alpha=0.85, edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=15, ha='right')
    ax.bar_label(bars, fmt='%.2f', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

In [ ]:
# Actual vs Predicted scatter (sample of 500)
preds_all = best_model.predict(X_test)
sample_idx = rng.choice(len(X_test), size=min(500, len(X_test)), replace=False)
y_samp, p_samp = y_test.values[sample_idx], preds_all[sample_idx]

plt.figure(figsize=(7, 6))
plt.scatter(y_samp, p_samp, alpha=0.4, s=15, color='#1976D2')
lims = [min(y_samp.min(), p_samp.min())-5, max(y_samp.max(), p_samp.max())+5]
plt.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted — {best_row["Model"]}')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    idx = np.argsort(imp)
    plt.figure(figsize=(8, 6))
    plt.barh([FEATURES[i] for i in idx], imp[idx],
             color=['#43A047' if imp[i] > np.median(imp) else '#90CAF9' for i in idx])
    plt.title('Feature Importance (Random Forest)', fontweight='bold')
    plt.xlabel('Importance Score')
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout(); plt.show()

## 7. Making Predictions

In [ ]:
# Example prediction
def predict_demand(route, hour, day_name, is_weekend, temperature,
                   weather, is_holiday, month, encoders, features, model):
    hour_sin  = np.sin(2*np.pi*hour/24)
    hour_cos  = np.cos(2*np.pi*hour/24)
    month_sin = np.sin(2*np.pi*month/12)
    month_cos = np.cos(2*np.pi*month/12)
    quarter   = (month-1)//3+1
    row = {
        'Route_ID'   : encoders['Route_ID'].transform([str(route)])[0],
        'Hour'       : hour, 'Hour_sin': hour_sin, 'Hour_cos': hour_cos,
        'Day_of_Week': encoders['Day_of_Week'].transform([str(day_name)])[0],
        'Is_Weekend' : int(is_weekend),
        'Temperature': temperature,
        'Weather'    : encoders['Weather'].transform([str(weather)])[0],
        'Is_Holiday' : int(is_holiday),
        'Month'      : month, 'Month_sin': month_sin, 'Month_cos': month_cos,
        'Quarter'    : quarter, 'WeekOfYear': 15,
    }
    X = pd.DataFrame([row])[features]
    return max(0, int(round(model.predict(X)[0])))

# Make a sample prediction
predicted = predict_demand(
    route='R03', hour=8, day_name='Monday', is_weekend=False,
    temperature=28, weather='Clear', is_holiday=False,
    month=3, encoders=encoders, features=FEATURES, model=best_model
)
print(f'Predicted passengers on R03 at 08:00 on a Monday (Clear, 28°C): {predicted}')

BUS_CAPACITY = 60
print(f'Buses needed: {max(1, int(np.ceil(predicted/BUS_CAPACITY)))}')

---
## Summary

| Step | Action |
|------|--------|
| Data | Generated ~65k synthetic records across 5 routes, 2 years |
| Clean | Removed duplicates, imputed missing values, capped outliers |
| Features | Cyclic time encoding, label encoding, temporal features |
| Split | Chronological 80/20 — no data leakage |
| Models | Baseline, Linear Regression, Decision Tree, Random Forest |
| Selection | Lowest RMSE on test set |
| Output | Trained model saved to `best_model.pkl` |

> ⚠️ **This is a prototype using synthetic data.** Real deployment requires real passenger data and proper validation.